# BayanBench: Ask Gemini (batched)

The free Gemini tier allows only about 20 requests per day, so each request carries **25 questions**. One question type per request, and never two questions about the same town in one request, so no question reveals another's answer.

Run all cells once a day. It stops by itself at the daily limit and continues from there the next day.

In [ ]:
!pip install -q google-genai

## 1. Settings

In [ ]:
MODEL = "gemini-3.6-flash"
SAMPLE = "pilot"        # first "pilot" (10 requests), then "core" (about 74 requests over several days)
BATCH_SIZE = 25
SECONDS_BETWEEN_REQUESTS = 15

import os, json
from google.colab import drive, files, userdata
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/bayanbench"
os.makedirs(SAVE_DIR, exist_ok=True)
RESULTS_FILE = f"{SAVE_DIR}/results_{MODEL}-batch{BATCH_SIZE}_{SAMPLE}.jsonl"
QUESTIONS_FILE = f"{SAVE_DIR}/questions_{SAMPLE}.jsonl"
if not os.path.exists(QUESTIONS_FILE):
    print(f"Please upload questions_{SAMPLE}.jsonl")
    up = files.upload()
    open(QUESTIONS_FILE, "wb").write(up[next(iter(up))])
questions = [json.loads(l) for l in open(QUESTIONS_FILE, encoding="utf-8")]
print(len(questions), "questions; results go to", RESULTS_FILE)

## 2. Batching code

In [ ]:
"""Batched question answering for Gemini (used by notebooks/03_ask_gemini_batched.ipynb).

The free tier allows very few requests per day, so each request carries a batch
of questions. Rules that keep batching from leaking answers:
  * a batch holds ONE question type only
  * a batch never holds two questions about the same town
Questions about a town's province or region never show its province, while
other question types do; keeping types and towns apart prevents one question
from revealing another's answer.
"""
import json
import random
import re
import time
import datetime

SYSTEM = """You answer factual questions about places in the Philippines.
Reply with JSON only, in this format: {"answer": <your answer>}
- For a province or a region, give its name as a string.
- For "a city or a municipality", answer "city" or "municipality".
- For a number of barangays, give a whole number.
- For "name three barangays", give a list of three barangay names.
If you are not sure, set the answer to "unknown"."""

# Same rules as the single-question prompt used for every other model;
# only the reply format differs (a list of answers with ids).
SYSTEM_BATCH = """You answer factual questions about places in the Philippines.
You will receive several numbered questions. Answer each one independently.
- For a province or a region, give its name as a string.
- For "a city or a municipality", answer "city" or "municipality".
- For a number of barangays, give a whole number.
- For "name three barangays", give a list of three barangay names.
If you are not sure, set the answer to "unknown".
Reply with JSON only, in this format:
{"answers": [{"id": "<question id>", "answer": <your answer>}, ...]}
Include every id exactly once."""


def make_batches(questions, size=25, seed=2026):
    """Deterministic batches: one question type per batch, no repeated town."""
    by_type = {}
    for q in questions:
        by_type.setdefault(q["qtype"], []).append(q)
    batches = []
    for qtype in sorted(by_type):
        qs = sorted(by_type[qtype], key=lambda q: q["qid"])
        random.Random(f"{seed}-{qtype}").shuffle(qs)
        for i in range(0, len(qs), size):
            batches.append(qs[i:i + size])
    for b in batches:
        assert len({q["lgu_code"] for q in b}) == len(b), "town repeated in a batch"
        assert len({q["qtype"] for q in b}) == 1, "mixed question types in a batch"
    return batches


def batch_prompt(batch):
    lines = [f'{q["qid"]}: {q["question"]}' for q in batch]
    return "Questions:\n" + "\n".join(lines)


def parse_batch(text):
    """Return {qid: answer} and an error message (None if fine)."""
    if not text:
        return {}, "empty response"
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    try:
        obj = json.loads(t)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", t, re.S)
        if not m:
            return {}, "not JSON"
        try:
            obj = json.loads(m.group(0))
        except json.JSONDecodeError:
            return {}, "not JSON"
    items = obj.get("answers") if isinstance(obj, dict) else obj
    if not isinstance(items, list):
        return {}, "no 'answers' list"
    out = {}
    for it in items:
        if isinstance(it, dict) and "id" in it and "answer" in it:
            out[str(it["id"]).strip()] = it["answer"]
    return out, None


class DailyQuotaReached(Exception):
    pass


def classify_error(msg):
    if "PerDay" in msg or "per day" in msg.lower():
        return "daily"
    if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
        return "rate"
    if "503" in msg or "UNAVAILABLE" in msg or "500" in msg:
        return "busy"
    return "other"


def run_batches(generate, model_name, questions, done, write, size=25,
                pause=15, max_requests=None, sleep=time.sleep, log=print):
    """generate(prompt_text) -> response text. write(record) saves one answer record.

    Returns the number of requests sent. Stops cleanly at the daily quota.
    """
    todo = [q for q in questions if q["qid"] not in done]
    batches = make_batches(todo, size)
    log(f"{len(todo)} questions to go in {len(batches)} requests")
    sent = 0
    for bi, batch in enumerate(batches, 1):
        if max_requests is not None and sent >= max_requests:
            log("Reached the request limit set for this run.")
            break
        text, err, wait = None, None, 30
        for attempt in range(3):
            try:
                sent += 1
                text = generate(batch_prompt(batch))
                err = None
                break
            except Exception as e:  # noqa: BLE001
                kind = classify_error(str(e))
                if kind == "daily":
                    raise DailyQuotaReached(str(e)[:200])
                err = str(e)[:300]
                if kind in ("rate", "busy") and attempt < 2:
                    log(f"  pause {wait}s ({kind})")
                    sleep(wait)
                    wait *= 2
                    continue
                break
        answers, perr = parse_batch(text) if err is None else ({}, None)
        now = datetime.datetime.now(datetime.timezone.utc).isoformat()
        got = 0
        for pos, q in enumerate(batch):
            rec = {"qid": q["qid"], "qtype": q["qtype"], "model": model_name, "mode": f"batch{size}",
                   "batch_index": bi, "batch_position": pos, "time": now}
            if err is not None:
                rec.update(raw=None, answer=None, parse_error=None, error=err)
            elif perr is not None:  # whole reply unreadable: a format failure, so retry next run
                rec.update(raw=text, answer=None, parse_error=None, error=f"unreadable batch reply: {perr}")
            elif q["qid"] in answers:
                rec.update(raw=None, answer=answers[q["qid"]], parse_error=None, error=None)
                got += 1
            else:  # model skipped this id: retry next run
                rec.update(raw=None, answer=None, parse_error=None, error="missing from batch reply")
            write(rec)
        reason = ""
        if err is not None:
            reason = f"  (failed: {classify_error(err)}; will retry next run)"
        elif perr is not None:
            reason = f"  (reply unreadable: {perr}; will retry next run)"
        elif got < len(batch):
            reason = "  (some ids missing; will retry next run)"
        log(f"request {bi}/{len(batches)} ({batch[0]['qtype']}): {got}/{len(batch)} answers{reason}")
        sleep(pause)
    return sent

## 3. Connect to Gemini (no test request, to save the daily quota)

In [ ]:
from google import genai
from google.genai import types
try:
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
CONFIG = types.GenerateContentConfig(system_instruction=SYSTEM_BATCH, temperature=0.0,
                                     response_mime_type="application/json")
MODEL_VERSION = {}

def generate(prompt):
    resp = client.models.generate_content(model=MODEL, contents=prompt, config=CONFIG)
    MODEL_VERSION["v"] = getattr(resp, "model_version", None)
    return resp.text

## 4. Run

In [ ]:
done = set()
if os.path.exists(RESULTS_FILE):
    for line in open(RESULTS_FILE, encoding="utf-8"):
        r = json.loads(line)
        if r.get("error") is None:
            done.add(r["qid"])

def write(rec):
    rec["model_version"] = MODEL_VERSION.get("v")
    with open(RESULTS_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

try:
    run_batches(generate, MODEL, questions, done, write, size=BATCH_SIZE,
                pause=SECONDS_BETWEEN_REQUESTS)
    ok = set()
    for line in open(RESULTS_FILE, encoding="utf-8"):
        r = json.loads(line)
        if r.get("error") is None:
            ok.add(r["qid"])
    left = len(questions) - len(ok)
    if left == 0:
        print("All questions in this sample are answered.")
    else:
        print(f"{left} questions still need an answer. Run this cell again to retry them.")
except DailyQuotaReached:
    print("Daily limit reached. Progress is saved. Run this notebook again tomorrow.")

## 5. Progress and download

In [ ]:
latest = {}
for line in open(RESULTS_FILE, encoding="utf-8"):
    r = json.loads(line)
    if r.get("error") is None:
        latest[r["qid"]] = r
print(f"{len(latest)} of {len(questions)} questions answered")
files.download(RESULTS_FILE)